# Load epub book

In [1]:
# Import libraries
import os
from langchain_community.document_loaders import UnstructuredEPubLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from uuid import uuid4
from chromadb.utils import embedding_functions

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
# TODO: Load document 
chunk_size = 1024
chunk_overlap = 102
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

epub_loader = UnstructuredEPubLoader('./docs/charles-dickens_a-christmas-carol.epub')

[WARNING] Could not load translations for en-US
  data file translations/en.yaml not found
[WARNING] The term Abstract has no translation defined.



In [ ]:
# TODO Split document
chunks = epub_loader.load_and_split(text_splitter)

In [4]:
# TODO Examine chunk
print(len(chunks))
print(chunks[100])

205
page_content='The house-fronts looked black enough, and the windows blacker, contrasting with the smooth white sheet of snow upon the roofs, and with the dirtier snow upon the ground; which last deposit had been ploughed up in deep furrows by the heavy wheels of carts and wagons: furrows that crossed and recrossed each other hundreds of times where the great streets branched off; and made intricate channels, hard to trace in the thick yellow mud and icy water. The sky was gloomy, and the shortest streets were choked up with a dingy mist, half thawed, half frozen, whose heavier particles descended in a shower of sooty atoms, as if all the chimneys in Great Britain had, by one consent, caught fire, and were blazing away to their dear heart’s content. There was nothing very cheerful in the climate or the town, and yet was there an air of cheerfulness abroad that the clearest summer air and brightest summer sun might have endeavoured to diffuse in vain.' metadata={'source': './docs/cha

# Create embeddings

In [5]:
# TODO: Create embedding model
embed_model_name = "BAAI/bge-small-en-v1.5"
#embed_model_name = "all-MiniLM-L6-v2"

chroma_embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embed_model_name)

In [6]:
# TODO: Explore embedding model
text = 'hello world'
emb_text = chroma_embed_func([ 'hello, world', 'big black bug bleeds black blood' ])


In [10]:
print(len(emb_text))
print(len(emb_text[0]))
print(len(emb_text[1]))
print(emb_text[0])

2
384
384
[-3.15818675e-02 -4.86476496e-02  3.21324095e-02 -6.57483190e-02
 -1.12417666e-03  1.14272060e-02 -1.62244460e-03  5.49600683e-02
  4.48704362e-02 -2.09960667e-03  7.87414052e-03 -2.20074598e-02
  3.43550555e-02  6.57045916e-02  2.98711844e-02 -2.77335406e-04
  1.02015398e-03 -3.47685143e-02 -1.21079251e-01 -1.47990324e-02
  9.72587019e-02  3.53695117e-02 -1.68968774e-02 -4.28635813e-02
 -2.48042475e-02  5.63809928e-03  6.80471864e-03  1.35493753e-02
  6.07592007e-03 -9.83635634e-02 -6.45543709e-02 -1.15323812e-02
  3.96090671e-02  2.41095200e-02  4.54739295e-02 -2.10404973e-02
  2.52140928e-02 -1.03885606e-02 -7.94328749e-02  3.64228617e-03
  4.60232161e-02 -5.09504005e-02  1.40664512e-02 -3.41335894e-03
  1.36136133e-02 -4.93411645e-02  1.70672331e-02  5.47222309e-02
 -2.78037973e-02  4.88183287e-04 -5.45995012e-02 -8.51241872e-03
 -1.97877828e-02 -2.24600383e-03  2.84831394e-02  9.09864828e-02
  7.97384828e-02  2.93898419e-03  4.68927287e-02  8.69192462e-03
  1.88648663e-0

In [12]:
# TODO: Prepare the chunks for inserting into Chroma
# Extract the text
texts = [ c.page_content for c in chunks ]
print(texts[100])
print(len(texts))


The house-fronts looked black enough, and the windows blacker, contrasting with the smooth white sheet of snow upon the roofs, and with the dirtier snow upon the ground; which last deposit had been ploughed up in deep furrows by the heavy wheels of carts and wagons: furrows that crossed and recrossed each other hundreds of times where the great streets branched off; and made intricate channels, hard to trace in the thick yellow mud and icy water. The sky was gloomy, and the shortest streets were choked up with a dingy mist, half thawed, half frozen, whose heavier particles descended in a shower of sooty atoms, as if all the chimneys in Great Britain had, by one consent, caught fire, and were blazing away to their dear heart’s content. There was nothing very cheerful in the climate or the town, and yet was there an air of cheerfulness abroad that the clearest summer air and brightest summer sun might have endeavoured to diffuse in vain.
205


In [14]:
text_ids = [  str(uuid4())[:8] for _ in range(len(texts))]
print(text_ids)
print(len(text_ids))

['416a997c', '939c9f94', '73429bae', '01597daf', 'a33593c0', '262f8825', '0729f0a5', 'd6942579', '1cf193eb', 'ef6e6bde', '8aee0e24', '4337d889', '79cae405', 'bef8c5b4', 'de1d5645', '75a21cc1', '8d78c99f', 'a743a6fd', '74c1a0ce', '5809e87f', '8144bf7d', '6f730e2a', '39cb2a9a', 'ebf315c5', 'fc84278f', '5614d4c4', '81a4ae6e', 'ec51b3cc', 'efcb1a3c', 'cabff4b0', 'aedec958', 'b387be5f', 'ebf06b97', '6d90fe73', '7579c6c4', 'fa0ea6a7', '55c29497', '3561f024', '4c13f38d', '0e6f2387', '5ee89024', '1478dbb2', '9d491aab', '04c63e85', '225a6237', '7d2c12ed', 'a4b563df', 'dc798219', 'b1d8d28b', '0716619c', '45acad02', '8e65fabe', '3d509c05', '1264893c', '91157b8a', 'b3f40de2', '609296ca', 'd9b42b7c', '07498739', 'ff62bcfe', '9e3db0bd', 'de7fe8bf', 'ce40f020', '82236369', 'f34acd70', 'f6071c95', 'acacab31', 'efce3a36', '1af3a493', 'a74dc23e', 'e6fd0557', '78b1c6a8', '03249927', '14ab9df3', '6f889053', '2806a201', 'fd494339', 'f553a776', '95b3b52f', '8444827c', '2592370a', 'd15294d2', '9192da51', 'ee

In [15]:
# TODO: Create ephemeral Chroma client and save chunks
col_name = 'carol'

# Create a the chromadb client
ch_client = chromadb.Client()

# drop the table
try:
   ch_client.delete_collection(col_name)
except:
   pass

# Insert the texts into the database
carol_col = ch_client.create_collection(
   name = col_name,
   embedding_function=chroma_embed_func
)


In [16]:
#Insert the docs into the collection
carol_col.add(
   documents = texts,
   ids = text_ids
)

In [17]:
# TODO: Print number of documents in collection 
print(carol_col.count())

205


In [ ]:
# TODO: Query collection 
query = "What happened Marley?"


results = carol_col.query(
   query_texts=[ query ],
   n_results=5
)

print(results)

{'ids': [['04c63e85', 'b1d8d28b', '01597daf', 'fc84278f', 'aedec958']], 'embeddings': None, 'documents': [['The apparition walked backward from him; and, at every step it took, the window raised itself a little, so that, when the spectre reached it, it was wide open. It beckoned Scrooge to approach, which he did. When they were within two paces of each other, Marley’s Ghost held up its hand, warning him to come no nearer. Scrooge stopped.\n\nNot so much in obedience as in surprise and fear; for, on the raising of the hand, he became sensible of confused noises in the air; incoherent sounds of lamentation and regret; wailings inexpressibly sorrowful and self-accusatory. The spectre, after listening for a moment, joined in the mournful dirge; and floated out upon the bleak, dark night.\n\nScrooge followed to the window: desperate in his curiosity. He looked out.', 'Marley’s Ghost bothered him exceedingly. Every time he resolved within himself, after mature inquiry that it was all a dream

In [23]:
for id in results['ids'][0]:
   result = carol_col.get(id)
   print(result['documents'])

['The apparition walked backward from him; and, at every step it took, the window raised itself a little, so that, when the spectre reached it, it was wide open. It beckoned Scrooge to approach, which he did. When they were within two paces of each other, Marley’s Ghost held up its hand, warning him to come no nearer. Scrooge stopped.\n\nNot so much in obedience as in surprise and fear; for, on the raising of the hand, he became sensible of confused noises in the air; incoherent sounds of lamentation and regret; wailings inexpressibly sorrowful and self-accusatory. The spectre, after listening for a moment, joined in the mournful dirge; and floated out upon the bleak, dark night.\n\nScrooge followed to the window: desperate in his curiosity. He looked out.']
['Marley’s Ghost bothered him exceedingly. Every time he resolved within himself, after mature inquiry that it was all a dream, his mind flew back again, like a strong spring released, to its first position, and presented the same 

# Question and Answer LLM
In this exercise you will implement a question and answer LLM for the 'A Christmas Carol' book that you have chunked and saved. 

The workflow is as follows:
1. Assume you ask the following question regarding the book eg. `"Who is Scrooge?"`?
2. Query the relevant context from Chroma with the question or facts from the question.
3. Combine the question and the top 5 context return by Chroma into a prompt 
4. Use `google/flan-t5-base` to answer the question.

Look through the FLAN templates in [Github](https://github.com/google-research/FLAN/blob/main/flan/templates.py) and select an appropriate template for this workshop.

Do not worry about the accuracy of the result. Focus on implementing the solution. We will discuss the nuances of the solution at the end of the workshop.

Use your RAG workflow to answer the provided questions in `questions_for_rag.txt` file. 

In [ ]:
# TODO Your code 

In [ ]:
# TODO Your code

In [ ]:
# TODO Your code

# Discussion

1. How did your solution perform?
2. Where do you think are the issues?
3. How can you improve it?